In [ ]:
from datetime import datetime
from pathlib import Path
from itertools import pairwise
from typing import Callable, Literal

import pandas as pd
import numpy as np
from scipy.integrate import trapezoid as integrate_trapezoid
import matplotlib.pyplot as plt
from airfoil import (
    Airfoil,
    Hole,
    Hinge,
    WingSegment,
    Decomposer,
    Wing
)
from airfoil.cnc import MachineSetup, CNC
from airfoil.wing import (
    angle_degrees_to_slope,
    mirror,
    auto_piecewise,
    auto_interpolate,

)
from airfoil.util import (
    resample_shapes,
    create_array_interpolator,
)
import pyvista as pv
XPS_FOAM_DENSITY = 40 # kg/m**3

In [ ]:
# wing half width

wing_sections_at = np.array([
    100.0, # 0
    200.0, # 1
    300.0, # 2
    400.0, # 3
    500.0, # 4
    600.0, # 5
    700.0, # 6
])
leading_edge = mirror(auto_piecewise([
    (wing_sections_at[0], lambda x: 0),
    (wing_sections_at[3], lambda x: -x*angle_degrees_to_slope(2)),
    (wing_sections_at[6], lambda x: -x*angle_degrees_to_slope(10)),
]))
trailing_edge = mirror(auto_piecewise([
    (wing_sections_at[0], lambda x: -200),
    (wing_sections_at[6], lambda x: -x*angle_degrees_to_slope( 2)),
]))
chord = lambda x: leading_edge(x)-trailing_edge(x)
dihedral = mirror(auto_piecewise([
    (wing_sections_at[0], lambda _: 0),
    (wing_sections_at[2], lambda x: x * angle_degrees_to_slope( 3)),
    (wing_sections_at[3], lambda x: x * angle_degrees_to_slope( 4)),
    (wing_sections_at[6], lambda x: x * angle_degrees_to_slope( 5)),
]))
washout = mirror(auto_interpolate([
    [                0.0,  4],
    [wing_sections_at[0],  4],
    [wing_sections_at[2], 0],
    [wing_sections_at[6], 0],
]))
hinge_line:Callable[[float],float] = mirror(lambda x: np.where(x>=wing_sections_at[2], trailing_edge(x) + chord(wing_sections_at[2]) * 0.3, np.nan))
spar_line :Callable[[float],float] = mirror(lambda x: np.where(x<=wing_sections_at[1],                  - chord(0)                   * 0.3, np.nan))
wing_airfoil = Airfoil.create_sampler(
    airfoil         = lambda x: Airfoil.from_naca_designation("4412", 100),
    leading_edge    = leading_edge,
    dihedral        = dihedral,
    chord           = chord,
    washout         = washout,
    rotation_center = lambda x: chord(x)*0.25
)

wing_segments_items:list[WingSegment] = []
for sla,slb in pairwise([-25]+list(wing_sections_at)):
    af1 = wing_airfoil(sla)
    af2 = wing_airfoil(slb)
    mid = (sla+slb)/2
    # if not np.isnan(hinge_line(mid)):
    #     af1 = af1.with_hinge(Hinge([-hinge_line(sla),0],angle_deg=10,rotation_deg=0),upper_thickness=2)
    #     af2 = af2.with_hinge(Hinge([-hinge_line(slb),0],angle_deg=10,rotation_deg=0),upper_thickness=2)
    # if not np.isnan(spar_line(mid)):
    #     af1 = af1.with_holes([Hole(diameter_mm=8, position=(-spar_line(sla),10))])
    #     af2 = af2.with_holes([Hole(diameter_mm=8, position=(-spar_line(slb),10))])
    wing_segments_items.append(WingSegment(
        left   = af1,
        right  = af2,
        length = slb-sla
    ))
wing_segments = Wing(
    segments=wing_segments_items,
    first_segment_is_central=True,
    mirrored=True,
)

In [ ]:
x_wing     = np.linspace(-700, 700, 400)
x_elevator = np.linspace(-150,150,400)
x_rudder = np.linspace(0,150,400)

fig, ax1 = plt.subplots(figsize=(5,8))
ax1.plot(x_wing,  leading_edge(x_wing))
ax1.plot(x_wing, trailing_edge(x_wing))
ax1.plot(x_wing, hinge_line(x_wing))
ax1.plot(x_wing, spar_line(x_wing))
ax1.set_aspect("equal")
for section in wing_sections_at:
    ax1.axvline(section,linestyle=":",c="r",linewidth=1)
    ax1.axvline(-section,linestyle=":",c="r",linewidth=1)

# ax1.plot(x_elevator, -400 -np.array([*map(lambda x:elevator_section(x).points[:,0].min(),x_elevator)]))
# ax1.plot(x_elevator, -400 -np.array([*map(lambda x:elevator_section(x).points[:,0].max(),x_elevator)]))
# ax1.plot(rudder_section(0).points[:,1],-400-rudder_section(0).points[:,0])

fig, ax2 = plt.subplots(figsize=(5,8))
sar = (-wing_sections_at)[::-1].tolist()+wing_sections_at.tolist()
ax2.plot(sar, np.array([wing_airfoil(xi).points[:,1].max()/100*chord(xi)+dihedral(xi) for xi in sar]))
ax2.plot(sar, np.array([wing_airfoil(xi).points[:,1].min()/100*chord(xi)+dihedral(xi) for xi in sar]))
# rl = np.array([*map(lambda x:rudder_section(x).points[:,1].min(),x_rudder)])
# rr = np.array([*map(lambda x:rudder_section(x).points[:,1].max(),x_rudder)])
# ax2.plot(rl,x_rudder-80)
# ax2.plot(rr,x_rudder-80)
# et = np.array([*map(lambda x:elevator_section(x).points[:,1].min(),x_elevator)])
# eb = np.array([*map(lambda x:elevator_section(x).points[:,1].max(),x_elevator)])
# ax2.plot(x_elevator,et-80)
# ax2.plot(x_elevator,eb-80)
ax2.set_aspect("equal")

In [ ]:
for seg in wing_segments.segments:
    de = Decomposer()
    fig,ax = plt.subplots(2, figsize=(8,3),sharex=True)
    seg.left.plot(ax=ax[0], decomposer=de)
    seg.right.plot(ax=ax[1], decomposer=de)

In [ ]:
pt = pv.Plotter()
wing_segment_meshes= wing_segments.to_meshes()#, add_mirrored=True, first_segment_is_central=True, share_decomposer=False)
for m in wing_segment_meshes:
    pt.add_mesh(
        m.rotate_x(-4).translate((0,0,60)),
        #pbr=True, 
        smooth_shading=True,
        split_sharp_edges=True,
        feature_angle=70,
        roughness=0.1,
        opacity=0.8,
    )
# elevator_segment_meshes = elevator_segments.to_meshes()#WingSegment.to_meshes(elevator_segments, add_mirrored=True, first_segment_is_central=True)
# for m in elevator_segment_meshes:
#     pt.add_mesh(
#         m.translate((0,400,0)),
#         #opacity=0.8,
#     )
# rudder_segment_meshes = rudder_segments.to_meshes()#WingSegment.to_meshes(rudder_segments, first_segment_is_central=False)
# for m in rudder_segment_meshes:
#     pt.add_mesh(
#         m.rotate_y(-90).translate((0,400,0)),
#         #opacity=0.8,
#     )
cl = 600
pt.add_mesh(pv.Cylinder((0,cl/2,0),direction=(0,1,0),radius=8,height=cl).translate((0,-180,0)))
pt.show()

In [ ]:


def round_special(excess:float=10, roundup:float=10):
    return lambda x: float(np.ceil(x/roundup)*roundup+excess)

def cut_size(ws:WingSegment, rounding:Callable[[float], float]=round_special()):
    bs = ws.bounding_size()
    depth = bs[0]
    width = bs[2]
    return float(width), rounding(depth)
def cut_list(
        wss:list[WingSegment],
        label:str|None=None,
        mode:Literal["mirrored", "mirrored centered", "as is"]="mirrored centered",
        rounding:Callable[[float], float] = round_special()
    ):
    df = pd.DataFrame([cut_size(item, rounding) for item in wss], columns=["width","depth"])
    match mode:
        case "mirrored":
            df = pd.concat([df,df])
        case "mirrored centered":
            df = pd.concat([df,df.iloc[1:]])
    res = df.groupby(["width","depth"]).size().rename("cut_list").to_frame()
    if label is not None:
        res["label"]=label
    return res
def cut_lists(ds:dict[str,tuple[list[WingSegment], Literal["mirrored", "mirrored centered", "as is"]]], rounding:Callable[[float],float]=round_special()):
    chunks = []
    for label, (segments, mode) in ds.items():
        chunks.append(cut_list(
            segments, 
            label=label, 
            mode=mode,
            rounding=rounding,
        ))
    return pd.concat(chunks).groupby(["width","depth"]).agg({"cut_list":"sum","label":lambda x: ", ".join(np.sort(np.unique(x)))})

rounding = round_special()
cl = cut_lists({
    "wing": (wing_segments.segments, "mirrored centered"),
    # "elevator": (elevator_segments.segments, "mirrored centered"),
    # "rudder": (rudder_segments.segments, "as is"),
}, rounding=rounding)
cl

In [ ]:
run_date = datetime.now()
airplane_name = "flappist"
file_prefix = f"{run_date:%Y-%m-%d} {airplane_name}"
run_folder = Path(f"./data/runs/{file_prefix}")
run_folder.mkdir(parents=True, exist_ok=True)
def make_filepath(suffix:str):
    return run_folder / f"{datetime.now():%H%M} {suffix}"
cl.to_csv(make_filepath("cut_list.csv"))
pt.export_gltf(make_filepath(f"{airplane_name}.gltf"))

In [ ]:
PLANE_SPACING = 232

In [ ]:
mirror = False
segment_index = 3
seg = wing_segments.segments[segment_index]
if mirror:
    seg = seg.with_mirror()
ms = MachineSetup(
    wing_segment       = seg.with_rotation(2),
    foam_height        = 30,
    foam_depth         = rounding(seg.bounding_size()[0]),
    plane_spacing      = PLANE_SPACING,
    decomposer         = Decomposer(buffer=0.5),
    max_cut_speed_mm_s = 180,
    min_cut_speed_mm_s = 130,
).with_recentered_part()
gcode = ms.prepare_gcode()
make_filepath(f"wing {segment_index}{' mirror' if mirror else ''}.json").write_text(
    ms.model_dump_json(indent=2)
)
print(f"length:{seg.length} x depth:{rounding(seg.bounding_size()[0])}\nlenght/2={seg.length/2}")
ms.plot()